# 建立一個 deployment

我們來為在 module 5 中建立的 `task_maistro` app 建立一個 deployment。

## 程式碼結構

[要建立一個 LangGraph Platform deployment，必須提供下列資訊](https://docs.langchain.com/langsmith/application-structure)：

* 一個 [LangGraph API 設定檔](https://docs.langchain.com/langsmith/application-structure#configuration-file) - `langgraph.json`
* 實作 application 邏輯的 graphs - 例如 `task_maistro.py`
* 一個指定執行 application 所需相依套件的檔案 - `requirements.txt`
* 提供 application 執行所需的環境變數 - `.env` 或 `docker-compose.yml`

這些我們在 `module-6/deployment` 目錄中都已經準備好了！

## CLI

[LangGraph CLI](https://docs.langchain.com/langsmith/cli) 是一個用來建立 LangGraph Platform deployment 的命令列介面。

In [1]:
%%capture --no-stderr
%pip install -U langgraph-cli

要建立一個 <!-- [~self-hosted deployment~](https://langchain-ai.github.io/langgraph/how-tos/deploy-self-hosted/#how-to-do-a-self-hosted-deployment-of-langgraph) --> [self-hosted deployment](https://docs.langchain.com/langsmith/self_hosted_data_plane)，我們會依循幾個步驟。

### 為 LangGraph Server 建置 Docker Image

我們先使用 langgraph CLI，為 [LangGraph Server](https://docs.google.com/presentation/d/18MwIaNR2m4Oba6roK_2VQcBE_8Jq_SI7VHTXJdl7raU/edit#slide=id.g313fb160676_0_32) 建立一個 Docker image。

這會把我們的 graph 與相依套件打包成一個 Docker image。

Docker image 是 Docker container 的範本，內含執行 application 所需的程式碼與相依套件。

請先確認已安裝 [Docker](https://docs.docker.com/engine/install/)，然後執行下列指令來建立 Docker image，`my-image`：

```
$ cd module-6/deployment
$ langgraph build -t my-image
```

### 設定 Redis 與 PostgreSQL

如果你已經有正在執行的 Redis 與 PostgreSQL（例如在本機或其他伺服器上），那麼可以搭配 Redis 與 PostgreSQL 的 URI，[單獨](https://docs.langchain.com/langsmith/deploy-hybrid#running-the-application-locally)建立並執行 LangGraph Server container：

```
docker run \
    --env-file .env \
    -p 8123:8000 \
    -e REDIS_URI="foo" \
    -e DATABASE_URI="bar" \
    -e LANGSMITH_API_KEY="baz" \
    my-image
```

或者，你也可以使用所提供的 `docker-compose.yml` 檔案，根據其中定義的服務建立三個獨立的 container：

* `langgraph-redis`：使用官方 Redis image 建立一個新的 container。
* `langgraph-postgres`：使用官方 Postgres image 建立一個新的 container。
* `langgraph-api`：使用你預先建置好的 image 建立一個新的 container。

只要複製 `docker-compose-example.yml`，並加入下列環境變數，即可執行部署好的 `task_maistro` app：

* `IMAGE_NAME`（例如 `my-image`） 
* `LANGSMITH_API_KEY`
* `OPENAI_API_KEY`

接著，<!-- [~launch the deployment~](https://langchain-ai.github.io/langgraph/how-tos/deploy-self-hosted/#using-docker-compose) [launch the deployment](https://docs.langchain.com/langsmith/self_hosted_data_plane): --> 啟動這個 deployment 吧！

```
$ cd module-6/deployment
$ docker compose up
```